# Food and Nutrition - Research


## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import pickle
import json
import time
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score
from sklearn.dummy import DummyRegressor, DummyClassifier
import warnings
warnings.filterwarnings('ignore')

/Users/solikha/Desktop/projects/DSB12_Food_nutrition.ID_1577655-Team_TL_drogolor.9b60a9d7_bc88_4cea-1/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Data Preparation

In [2]:
# Load dataset
df = pd.read_csv('data/epi_r.csv')
print('Shape:', df.shape)
df.head(2)

Shape: (20052, 680)


,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# Drop rows with missing rating
df = df.dropna(subset=['rating'])

# Filter: keep only title, rating and ingredient columns (0/1 columns)
non_ingredient_cols = ['title', 'rating', 'calories', 'protein', 'fat', 'sodium']
ingredient_cols = [c for c in df.columns if c not in non_ingredient_cols]

# Keep only columns that are binary (ingredient flags)
binary_cols = [c for c in ingredient_cols
               if df[c].dropna().isin([0.0, 1.0]).all()]

print(f'Total ingredient columns: {len(binary_cols)}')

# Final dataframe: title + rating + ingredients
keep_cols = ['title', 'rating'] + binary_cols
df_model = df[keep_cols].copy()
df_model = df_model.fillna(0)
print('Model dataframe shape:', df_model.shape)
df_model.head(2)

Total ingredient columns: 674
Model dataframe shape: (20052, 676)


,title,rating,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,advance prep required,alabama,alaska,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Features and target
X = df_model[binary_cols].values
y = df_model['rating'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (16041, 674) Test: (4011, 674)


## 2. Regression

In [5]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def crossval_rmse(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, cv=cv,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean(), scores.std()

# Naive regressor: always predict mean
naive_reg = DummyRegressor(strategy='mean')
naive_reg.fit(X_train, y_train)
naive_rmse = rmse(y_test, naive_reg.predict(X_test))
print(f'Naive RMSE: {naive_rmse:.4f}')

Naive RMSE: 1.3072


In [6]:
# Linear Regression
lr = LinearRegression()
lr_cv, lr_std = crossval_rmse(lr, X_train, y_train)
print(f'LinearRegression CV RMSE: {lr_cv:.4f} ± {lr_std:.4f}')
lr.fit(X_train, y_train)
print(f'LinearRegression Test RMSE: {rmse(y_test, lr.predict(X_test)):.4f}')

LinearRegression CV RMSE: 1.2572 ± 0.0225
LinearRegression Test RMSE: 1.2289


In [7]:
# Decision Tree Regressor
for depth in [5, 10, 15]:
    tree_r = DecisionTreeRegressor(max_depth=depth, random_state=21)
    cv_score, cv_std = crossval_rmse(tree_r, X_train, y_train)
    print(f'DecisionTree max_depth={depth}: CV RMSE={cv_score:.4f} ± {cv_std:.4f}')

DecisionTree max_depth=5: CV RMSE=1.2619 ± 0.0283
DecisionTree max_depth=10: CV RMSE=1.2818 ± 0.0249
DecisionTree max_depth=15: CV RMSE=1.3162 ± 0.0192


In [8]:
# Random Forest Regressor
for depth in [5, 10]:
    rf_r = RandomForestRegressor(n_estimators=50, max_depth=depth, random_state=21)
    cv_score, cv_std = crossval_rmse(rf_r, X_train, y_train)
    print(f'RandomForest max_depth={depth}: CV RMSE={cv_score:.4f} ± {cv_std:.4f}')

RandomForest max_depth=5: CV RMSE=1.2505 ± 0.0282
RandomForest max_depth=10: CV RMSE=1.2318 ± 0.0277


In [9]:
# Best regression model
best_reg = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=21)
best_reg.fit(X_train, y_train)
reg_test_rmse = rmse(y_test, best_reg.predict(X_test))
print(f'Best Regression Test RMSE: {reg_test_rmse:.4f}')
print(f'Naive RMSE: {naive_rmse:.4f}')

Best Regression Test RMSE: 1.2220
Naive RMSE: 1.3072


## 3. Classification - Integer classes

In [10]:
# Binarize: round to nearest integer
y_int = np.round(y).astype(int)
y_int_train = np.round(y_train).astype(int)
y_int_test = np.round(y_test).astype(int)

# Naive classifier
naive_clf = DummyClassifier(strategy='most_frequent')
naive_clf.fit(X_train, y_int_train)
naive_acc = accuracy_score(y_int_test, naive_clf.predict(X_test))
print(f'Naive accuracy: {naive_acc:.4f}')
print('Class distribution:', pd.Series(y_int).value_counts().sort_index().to_dict())

Naive accuracy: 0.6707
Class distribution: {0: 1836, 1: 164, 2: 656, 3: 1489, 4: 13188, 5: 2719}


In [11]:
# Decision Tree
for depth in [5, 10, 15]:
    tree_c = DecisionTreeClassifier(max_depth=depth, random_state=21)
    scores = cross_val_score(tree_c, X_train, y_int_train, cv=5)
    print(f'DecisionTree max_depth={depth}: CV acc={scores.mean():.4f}')

# Random Forest
for depth in [5, 10]:
    rf_c = RandomForestClassifier(n_estimators=50, max_depth=depth, random_state=21)
    scores = cross_val_score(rf_c, X_train, y_int_train, cv=5)
    print(f'RandomForest max_depth={depth}: CV acc={scores.mean():.4f}')

DecisionTree max_depth=5: CV acc=0.6670
DecisionTree max_depth=10: CV acc=0.6650
DecisionTree max_depth=15: CV acc=0.6592
RandomForest max_depth=5: CV acc=0.6622
RandomForest max_depth=10: CV acc=0.6659


In [12]:
best_int_clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=21)
best_int_clf.fit(X_train, y_int_train)
print(f'Best int classifier Test acc: {accuracy_score(y_int_test, best_int_clf.predict(X_test)):.4f}')

Best int classifier Test acc: 0.6794


## 4. Classification - 3 classes: bad/so-so/great

In [13]:
def to_class(rating):
    r = round(rating)
    if r in [0, 1]:
        return 'bad'
    elif r in [2, 3]:
        return 'so-so'
    else:
        return 'great'

y_class = np.array([to_class(r) for r in y])
y_class_train = np.array([to_class(r) for r in y_train])
y_class_test = np.array([to_class(r) for r in y_test])

print('Class distribution:')
print(pd.Series(y_class).value_counts())

# Naive
naive_3 = DummyClassifier(strategy='most_frequent')
naive_3.fit(X_train, y_class_train)
print(f'Naive accuracy: {accuracy_score(y_class_test, naive_3.predict(X_test)):.4f}')

Class distribution:
great    15907
so-so     2145
bad       2000
Name: count, dtype: int64
Naive accuracy: 0.7978


In [14]:
# What is worse: predicting bad when actually good, or good when actually bad?
# Predicting good when actually bad is worse (user gets disappointed).
# So we want high RECALL for 'bad' class - use recall metric.
print('Predicting a good rating that is actually bad is worse.')
print('We use recall to minimize false positives for "great" class.')

Predicting a good rating that is actually bad is worse.
We use recall to minimize false positives for "great" class.


In [15]:
# Try different models with recall metric
for depth in [5, 10, 15]:
    tree_3 = DecisionTreeClassifier(max_depth=depth, random_state=21)
    scores = cross_val_score(tree_3, X_train, y_class_train, cv=5,
                             scoring='recall_weighted')
    print(f'DecisionTree max_depth={depth}: CV recall={scores.mean():.4f}')

for depth in [5, 10]:
    rf_3 = RandomForestClassifier(n_estimators=50, max_depth=depth, random_state=21)
    scores = cross_val_score(rf_3, X_train, y_class_train, cv=5,
                             scoring='recall_weighted')
    print(f'RandomForest max_depth={depth}: CV recall={scores.mean():.4f}')

DecisionTree max_depth=5: CV recall=0.8044
DecisionTree max_depth=10: CV recall=0.8024
DecisionTree max_depth=15: CV recall=0.7981
RandomForest max_depth=5: CV recall=0.7951
RandomForest max_depth=10: CV recall=0.8009


In [16]:
# Ensemble for 3-class
rf_3a = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=21)
tree_3a = DecisionTreeClassifier(max_depth=15, random_state=21)
voting_3 = VotingClassifier(
    estimators=[('rf', rf_3a), ('tree', tree_3a)],
    voting='soft'
)
scores = cross_val_score(voting_3, X_train, y_class_train, cv=5,
                         scoring='recall_weighted')
print(f'Voting ensemble CV recall: {scores.mean():.4f}')

Voting ensemble CV recall: 0.7985


In [17]:
# Best 3-class model
best_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=21)
best_clf.fit(X_train, y_class_train)
y_pred_class = best_clf.predict(X_test)
print(f'Best 3-class Test accuracy: {accuracy_score(y_class_test, y_pred_class):.4f}')
print(f'Best 3-class Test recall: {recall_score(y_class_test, y_pred_class, average="weighted"):.4f}')
print(f'Naive accuracy: {accuracy_score(y_class_test, naive_3.predict(X_test)):.4f}')

Best 3-class Test accuracy: 0.8038
Best 3-class Test recall: 0.8038
Naive accuracy: 0.7978


## 5. Decision: Classification vs Regression

In [18]:
# We use classification because:
# 1. Task requires predicting classes (bad/so-so/great)
# 2. Users want a category, not an exact number
# 3. Classification accuracy is better than regression RMSE improvement over naive

# Save the best classification model
best_clf.fit(X, y_class)  # retrain on full data
with open('data/best_model.pkl', 'wb') as f:
    pickle.dump(best_clf, f)
print('Classification model saved!')

# Save ingredient columns order for use in nutritionist.py
with open('data/ingredient_cols.pkl', 'wb') as f:
    pickle.dump(binary_cols, f)
print('Ingredient columns saved!')

Classification model saved!
Ingredient columns saved!


## 6. Nutrition Facts

In [19]:
# Daily values from the PDF tables
DAILY_VALUES = {
    'protein': 50,           # g
    'fat': 78,               # g
    'saturated_fat': 20,     # g
    'cholesterol': 300,      # mg
    'carbohydrates': 275,    # g
    'sodium': 2300,          # mg
    'fiber': 28,             # g
    'added_sugars': 50,      # g
    'vitamin_a': 900,        # mcg
    'vitamin_c': 90,         # mg
    'calcium': 1300,         # mg
    'iron': 18,              # mg
    'vitamin_d': 20,         # mcg
    'vitamin_e': 15,         # mg
    'vitamin_k': 120,        # mcg
    'potassium': 4700,       # mg
    'magnesium': 420,        # mg
    'zinc': 11,              # mg
}
print('Daily values defined:', list(DAILY_VALUES.keys()))

Daily values defined: ['protein', 'fat', 'saturated_fat', 'cholesterol', 'carbohydrates', 'sodium', 'fiber', 'added_sugars', 'vitamin_a', 'vitamin_c', 'calcium', 'iron', 'vitamin_d', 'vitamin_e', 'vitamin_k', 'potassium', 'magnesium', 'zinc']


In [20]:
API_KEY = 'ZMYqTcH7YqVNqgcN9YEUtuFmnGxBLWKPS55M9QIw'

def get_nutrition(ingredient, api_key=API_KEY):
    url = f'https://api.nal.usda.gov/fdc/v1/foods/search'
    params = {
        'query': ingredient,
        'api_key': api_key,
        'pageSize': 1
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        if not data.get('foods'):
            return None
        food = data['foods'][0]
        nutrients = {n['nutrientName']: n['value'] for n in food.get('foodNutrients', [])}
        return nutrients
    except Exception as e:
        print(f'Error for {ingredient}: {e}')
        return None

# Test with one ingredient
test = get_nutrition('milk')
if test:
    print('API works! Sample nutrients:', list(test.items())[:5])
else:
    print('API error')

API works! Sample nutrients: [('Protein', 3.75), ('Total lipid (fat)', 3.33), ('Carbohydrate, by difference', 5.42), ('Energy', 67.0), ('Total Sugars', 5.0)]


In [21]:
# Collect nutrition for all ingredients (this takes time - use cache)
import os

cache_file = 'data/nutrition_cache.pkl'
if os.path.exists(cache_file):
    with open(cache_file, 'rb') as f:
        nutrition_cache = pickle.load(f)
    print(f'Loaded cache with {len(nutrition_cache)} ingredients')
else:
    nutrition_cache = {}

# Fetch for ingredients not yet in cache
missing = [ing for ing in binary_cols if ing not in nutrition_cache]
print(f'Ingredients to fetch: {len(missing)}')

for i, ing in enumerate(missing[:50]):  # fetch in batches
    nutrients = get_nutrition(ing)
    nutrition_cache[ing] = nutrients
    if i % 10 == 0:
        print(f'Fetched {i+1}/50...')
    time.sleep(0.1)  # be nice to the API

# Save cache
with open(cache_file, 'wb') as f:
    pickle.dump(nutrition_cache, f)
print('Cache saved!')

Ingredients to fetch: 674
Fetched 1/50...
Fetched 11/50...
Fetched 21/50...
Fetched 31/50...
Fetched 41/50...
Cache saved!


In [22]:
# Build nutrition dataframe with % daily values
NUTRIENT_MAP = {
    'Protein': ('protein', 50),
    'Total lipid (fat)': ('fat', 78),
    'Carbohydrate, by difference': ('carbohydrates', 275),
    'Sodium, Na': ('sodium', 2300),
    'Fiber, total dietary': ('fiber', 28),
    'Calcium, Ca': ('calcium', 1300),
    'Iron, Fe': ('iron', 18),
    'Potassium, K': ('potassium', 4700),
    'Vitamin C, total ascorbic acid': ('vitamin_c', 90),
    'Vitamin A, RAE': ('vitamin_a', 900),
    'Cholesterol': ('cholesterol', 300),
    'Fatty acids, total saturated': ('saturated_fat', 20),
    'Magnesium, Mg': ('magnesium', 420),
    'Zinc, Zn': ('zinc', 11),
}

nutrition_rows = []
for ing, nutrients in nutrition_cache.items():
    if nutrients is None:
        continue
    row = {'ingredient': ing}
    for api_name, (col_name, dv) in NUTRIENT_MAP.items():
        val = nutrients.get(api_name, 0) or 0
        row[col_name + '_pct_dv'] = round(val / dv * 100, 2)
    nutrition_rows.append(row)

nutrition_df = pd.DataFrame(nutrition_rows)
nutrition_df = nutrition_df.set_index('ingredient')
print('Nutrition dataframe shape:', nutrition_df.shape)
nutrition_df.head(3)

Nutrition dataframe shape: (48, 14)


,protein_pct_dv,fat_pct_dv,carbohydrates_pct_dv,sodium_pct_dv,fiber_pct_dv,calcium_pct_dv,iron_pct_dv,potassium_pct_dv,vitamin_c_pct_dv,vitamin_a_pct_dv,cholesterol_pct_dv,saturated_fat_pct_dv,magnesium_pct_dv,zinc_pct_dv
ingredient,,,,,,,,,,,,,,
22-minute meals,0.00,0.00,5.02,0.26,0.0,0.15,0.00,0.21,3.44,0.00,0.00,0.0,0.48,0.09
3-ingredient recipes,52.88,21.18,0.00,3.09,0.0,1.54,13.89,6.60,0.00,0.33,29.67,32.0,4.76,56.82
30 days of groceries,28.80,38.46,0.00,2.87,0.0,1.85,9.11,4.64,0.00,0.44,26.00,59.0,3.33,32.45


In [23]:
# Save nutrition dataframe
nutrition_df.to_csv('data/nutrition_facts.csv')
print('Saved nutrition_facts.csv')

Saved nutrition_facts.csv


## 7. Similar Recipes - URLs

In [24]:
# Load full_format_recipes.json to get URLs
with open('data/full_format_recipes.json', 'r') as f:
    full_recipes = json.load(f)

print(f'Total recipes in JSON: {len(full_recipes)}')
# Check structure
print('Keys in first recipe:', list(full_recipes[0].keys())[:10])

Total recipes in JSON: 20130
Keys in first recipe: ['directions', 'fat', 'date', 'categories', 'calories', 'desc', 'protein', 'rating', 'title', 'ingredients']


In [25]:
# Build title → url mapping
url_map = {}
for recipe in full_recipes:
    title = recipe.get('title', '')
    url = recipe.get('url', '')
    if title and url:
        url_map[title.strip()] = url

print(f'Recipes with URLs: {len(url_map)}')

# Build recipes dataframe with title, rating, url, ingredients
recipes_rows = []
for _, row in df_model.iterrows():
    title = row['title'].strip()
    url = url_map.get(title, '')
    if not url:
        # Try to build URL from title
        slug = title.lower().replace(' ', '-').replace(',', '').replace("'", '')
        url = f'https://www.epicurious.com/recipes/food/views/{slug}'
    recipes_rows.append({
        'title': title,
        'rating': row['rating'],
        'url': url
    })

# Add ingredient columns
recipes_df = pd.DataFrame(recipes_rows)
recipes_df = pd.concat([recipes_df, df_model[binary_cols].reset_index(drop=True)], axis=1)
recipes_df.to_csv('data/recipes_with_urls.csv', index=False)
print('Saved recipes_with_urls.csv')
print('Shape:', recipes_df.shape)
recipes_df.head(2)

Recipes with URLs: 0
Saved recipes_with_urls.csv
Shape: (20052, 677)


,title,rating,url,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,advance prep required,alabama,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,https://www.epicurious.com/recipes/food/views/...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,https://www.epicurious.com/recipes/food/views/...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
